# Test With External LangGraph Agent

This notebook uses a LangGraph agent outside the MCP server, calls the MCP `plot_sine_wave` tool with `num_points=30`, and displays the generated PNG inline.

## 0. Setup

Run this once. It installs the starter, the demo science package, and the notebook/agent dependencies into the active notebook kernel.

In VS Code, after this finishes, use **Select Kernel** and choose the Python interpreter printed by this cell if VS Code prompts you.

In [ ]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        str(ROOT / "science-demo"),
        "-e",
        str(ROOT),
        "jupyter",
        "ipykernel",
        "langgraph",
        "langchain-openai",
        "langchain-mcp-adapters",
    ]
)
print(sys.executable)

## 1. Pick One LLM Backend

Run one of the next two cells. The OpenAI cell is the easiest path.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Optional if OPENAI_API_KEY is not already exported in your shell:
# os.environ["OPENAI_API_KEY"] = "sk-..."

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5.5"))
llm

In [ ]:
from langchain_openai import ChatOpenAI

# Optional local backend. Run this only if an OpenAI-compatible local server is already running.
llm = ChatOpenAI(
    model="local-tool-model",
    base_url="http://127.0.0.1:8001/v1",
    api_key="not-needed",
    use_responses_api=False,
)
llm

## 2. Initialize MCP Tools

In [ ]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "science": {
            "transport": "stdio",
            "command": sys.executable,
            "args": ["-m", "fast_mcp_starter", "--transport", "stdio"],
            "cwd": str(ROOT),
        }
    }
)

tools = await mcp_client.get_tools()
[tool.name for tool in tools]

## 3. Initialize LangGraph Agent

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools)
agent

## 4. Ask The Agent To Plot 30 Points

In [ ]:
from IPython.display import Image, display
from langchain_core.messages import ToolMessage

output_dir = ROOT / "agent-output"
output_dir.mkdir(exist_ok=True)

result = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f"Use plot_sine_wave with output_dir={output_dir} and num_points=30.",
            }
        ]
    }
)

structured = [
    message.artifact["structured_content"]
    for message in result["messages"]
    if isinstance(message, ToolMessage) and message.artifact
]
plot_path = Path(structured[-1]["files"][0])
display(Image(filename=str(plot_path)))